# 대화 질문을 독립 검색어로 재작성하기

이 노트북은 공개 PDF 6건과 Ch11의 30문항을 사용합니다. 첫 셀부터 순서대로 실행합니다.

질문 재작성용 `gpt-4.1-mini`는 Ch6·Ch9의 답변 생성용 `gpt-5.4-mini`와 과제가 다릅니다. 원문·재작성문 검색을 비교하며 모델 간 답변 품질은 비교하지 않습니다.


## 입력과 API 키 준비

프로젝트 루트의 .env를 읽고 키의 존재만 확인합니다. 실제 값은 출력하지 않습니다.

### 재작성 모델의 입력과 출력 경계

대화 기록과 후속 질문을 프롬프트에 넣고 `model.invoke()`를 호출할 때 검색용 한 문장 생성이 실행됩니다. 재작성은 빠진 대상을 보충할 수 있지만 조건을 삭제하거나 추가할 수도 있습니다. 원문 검색 결과를 버리지 않고 두 순위를 RRF로 합쳐 변형 위험을 관찰합니다. 이번 출력은 원문·재작성·RRF 모두 E01을 상위에 포함하므로 검색 개선은 관찰하지 못했습니다.


In [1]:
from pathlib import Path
import json
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "labs/part3/ch12-retrieval-tuning").exists())
data = json.loads((root / "labs/part3/ch12-retrieval-tuning/data/advanced-documents.json").read_text())
queries = json.loads((root / "labs/part3/ch12-retrieval-tuning/data/advanced-queries.json").read_text())
chunks, parents = data["chunks"], {item["doc_id"]: item for item in data["parents"]}
vectorizer = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4), min_df=1)
matrix = vectorizer.fit_transform([item["text"] for item in chunks])
print(f"부모 문서 {len(parents)}건 · 자식 청크 {len(chunks)}개 · 평가 질문 {len(queries)}개")
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
load_dotenv(root / ".env")
print("OPENAI_API_KEY 준비:", bool(__import__("os").getenv("OPENAI_API_KEY")))

부모 문서 6건 · 자식 청크 175개 · 평가 질문 30개


OPENAI_API_KEY 준비: True


## 실제 모델 호출

대화 맥락이 필요한 후속 질문을 문서 검색용 한 문장으로 바꿉니다.


In [2]:
model = init_chat_model("openai:gpt-4.1-mini", temperature=0)
prompt = """다음 대화의 마지막 질문을 독립적인 한국어 검색 질문 한 문장으로 바꾸세요.
설명이나 따옴표를 덧붙이지 마세요.
사용자: 미르시립도서관 이용 일정을 살펴보고 있어요.
사용자: 그곳의 정기 휴관일은 언제예요?"""
response = model.invoke(prompt)
rewritten = response.text.strip()
print("재작성 결과:", rewritten)

재작성 결과: 미르시립도서관 정기 휴관일은 언제인가요?


## 원문과 재작성 질문의 검색 결과 비교

검색 함수는 그대로 두고 질문 문자열만 바꿉니다.


In [3]:
def top_parents(question, k=5):
    scores = cosine_similarity(vectorizer.transform([question]), matrix)[0]
    return [chunks[idx]["parent_id"] for idx in np.argsort(-scores)[:k]]

original = "그곳의 정기 휴관일은 언제예요?"
print("원문     :", top_parents(original))
print("재작성문 :", top_parents(rewritten))

원문     : ['E01', 'E02', 'M02', 'E02', 'E02']
재작성문 : ['E01', 'E01', 'E02', 'E01', 'E01']


## 원문과 재작성문의 순위 결합

두 질문을 따로 검색한 뒤 RRF로 결합합니다. 원문 결과를 지우지 않으므로 재작성에서 빠진 표현도 후보에 남길 수 있습니다.


In [4]:
def ranked_indices(question, k=10):
    scores = cosine_similarity(vectorizer.transform([question]), matrix)[0]
    return list(np.argsort(-scores)[:k])

rrf_scores = {}
for ranking in (ranked_indices(original), ranked_indices(rewritten)):
    for rank, idx in enumerate(ranking, 1):
        rrf_scores[idx] = rrf_scores.get(idx, 0.0) + 1 / (60 + rank)
fused = sorted(rrf_scores, key=rrf_scores.get, reverse=True)[:5]
print("RRF 결합 :", [chunks[idx]["parent_id"] for idx in fused])

RRF 결합 : ['E01', 'E02', 'E02', 'M02', 'E01']


## 모든 질문을 재작성할 필요는 없습니다

이미 기관명과 대상이 있는 개발용 Q001에는 재작성을 추가하지 않아도 되는지 비교합니다.


In [5]:
standalone_item = next(item for item in queries if item["qid"] == "Q001")
assert standalone_item["split"] == "dev"
standalone = standalone_item["question"]
print("독립 질문:", standalone)
print("검색 결과:", top_parents(standalone))
assert "미르시립도서관" in standalone

독립 질문: 미르시립도서관의 정기 휴관일은 언제인가요?
검색 결과: ['E01', 'E01', 'E02', 'E01', 'E01']
